In [ ]:
# 사전 설정

In [5]:
# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")


✅ OpenAI 클라이언트 준비 완료


In [2]:
# 🔧 LLM 호출 헬퍼 함수 — 앞으로 계속 사용합니다
def ask_llm(messages, temperature=0.3, max_tokens=500):
    """메시지 리스트를 받아서 LLM의 답변 텍스트를 돌려준다."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


In [ ]:
# 문제 1번

In [6]:
# 🎯 후보 system 메시지 3개 — 셋 다 돌려보고 비교
candidates = {
    "A": "회의록을 요약하세요.",
    "B": "당신은 회의록 요약 비서입니다. 3줄로 요약하세요.",
    "C": ("당신은 회의록 요약 비서입니다. 정확히 3줄로 요약하되, "
          "각 줄은 (결정사항) / (담당자) / (마감일) 형식을 따르세요. "
          "회의록에 없는 내용은 절대 추측하지 마세요."),
}

sample_notes = """
[2026-05-14 팀 미팅]
- 김PM: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안. 이유는 평가 데이터 부족.
- 이개발: 6월 15일이면 가능. 다만 RAG 평가셋 100건 추가 수집 필요.
- 박마케팅: 출시 연기 시 광고 일정도 조정해야 함. 다음 주 월요일까지 결정 필요.
"""

# 세 후보를 모두 실행해서 비교
for key, system_msg in candidates.items():
    result = ask_llm([
        {"role": "system", "content": system_msg},
        {"role": "user", "content": sample_notes},
    ], temperature=0)
    print(f"=== 후보 {key} ===")
    print(result)
    print()

=== 후보 A ===
**회의록 요약 (2026-05-14 팀 미팅)**

- **김PM**: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안, 이유는 평가 데이터 부족.
- **이개발**: 6월 15일 출시 가능, 그러나 RAG 평가셋 100건 추가 수집 필요.
- **박마케팅**: 출시 연기 시 광고 일정 조정 필요, 다음 주 월요일까지 결정 필요.

=== 후보 B ===
김PM은 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기할 것을 제안했으며, 이개발은 추가 데이터 수집이 필요하다고 언급했습니다. 박마케팅은 출시 연기에 따라 광고 일정 조정이 필요하다고 강조하며, 다음 주 월요일까지 결정을 요구했습니다.

=== 후보 C ===
(신규 챗봇 출시일을 6월 15일로 연기) / (김PM, 이개발, 박마케팅) / (다음 주 월요일)



In [ ]:
# 후보 C가 필요한 내용만 요약하여 보기 좋게 정리하였기에 제일 좋은 비서라고 사료됨. 
# 하지만 **이개발**이 말했던 RAG 평가셋 100건 추가 수집에 대한 정보가 들어가 있지 않아 모든 정보를 다 보여주는 지에 대한 확인이 필요함.

In [ ]:
# 문제 2번

In [15]:
def classify_inquiry(text: str) -> str:
    """고객 문의를 4개 카테고리 중 하나로 분류"""
    # 👇 여기를 채워보세요
    messages = [
        {"role": "system", "content": "너는 쇼핑몰 CS 담당자야. 문의를 읽고 결제, 배송, 반품, 기타 중 하나로만 분류해."},  # ← 빈칸
        {"role": "user", "content": text},
    ]
    return ask_llm(messages, temperature=0, max_tokens=10)


# 🎯 실행 *전* 예측: 아래 4개 문의가 어떻게 분류될지 머릿속으로 먼저 답을 적어보세요
test_cases = [
    "어제 주문했는데 카드 결제가 두 번 되었어요.",       # 예측: 결제
    "택배가 일주일째 안 와요.",                          # 예측: 배송
    "사이즈가 안 맞아서 반품하고 싶어요.",                # 예측: 반품
    "영업시간이 어떻게 되나요?",                         # 예측: 기타
]

# 이제 실행해서 예측과 비교
for text in test_cases:
    category = classify_inquiry(text)
    print(f"[{category:>2}] {text}")

[결제] 어제 주문했는데 카드 결제가 두 번 되었어요.
[배송] 택배가 일주일째 안 와요.
[반품] 사이즈가 안 맞아서 반품하고 싶어요.
[기타] 영업시간이 어떻게 되나요?


In [ ]:
# 빈칸: 
    # 너는 쇼핑몰 CS 담당자야. 문의를 보고 카테고리만 답해줘.

# 예측: 중복 결제
# 예측: 배송 지연
# 예측: 반품
# 예측: 영업 시간 확인